In [1]:
import os
import glob
import os
from scipy.io import mmwrite
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import yaml
import time
import gget
import psutil
from scipy.stats import zscore

import rmm
import cupy as cp
from rmm.allocators.cupy import rmm_cupy_allocator
import anndata as an
import scanpy as sc
import rapids_singlecell as rsc
import scvi

sc.settings.verbosity = 3

/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


In [2]:
# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

In [3]:
%%time
fpath = "/scratch/indikar_root/indikar1/shared_data/hematokytos/processed/sample_1_adata.h5ad"
adata = sc.read_h5ad(fpath)
adata

CPU times: user 4.34 s, sys: 31 s, total: 35.3 s
Wall time: 1min 7s


AnnData object with n_obs × n_vars = 1125041 × 52164
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'X_umap_X_scANVI', 'X_umap_raw_data', 'X_umap_scVI', '_scvi_manager_uuid', '_scvi_uuid', 'basename_colors', 'basename_palette

In [4]:
rsc.get.anndata_to_GPU(adata) # move to GPU
rsc.pp.filter_genes(adata, min_counts=500)
adata

filtered out 23626 genes that are detected in less than 500 counts


AnnData object with n_obs × n_vars = 1125041 × 28538
    obs: 'soma_joinid', 'dataset_id', 'assay', 'assay_ontology_term_id', 'cell_type', 'cell_type_ontology_term_id', 'development_stage', 'development_stage_ontology_term_id', 'disease', 'disease_ontology_term_id', 'donor_id', 'is_primary_data', 'observation_joinid', 'self_reported_ethnicity', 'self_reported_ethnicity_ontology_term_id', 'sex', 'sex_ontology_term_id', 'suspension_type', 'tissue', 'tissue_ontology_term_id', 'tissue_type', 'tissue_general', 'tissue_general_ontology_term_id', 'raw_sum', 'nnz', 'raw_mean_nnz', 'raw_variance_nnz', 'n_measured_vars', 'basename', 'dataset_id_int', 'n_counts', 'n_genes', '_scvi_batch', '_scvi_labels', 'leiden'
    var: 'n_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_nbatches', 'highly_variable_intersection'
    uns: 'X_umap_X_scANVI', 'X_umap_raw_data', 'X_umap_scVI', '_scvi_manager_uuid', '_scvi_uuid', 'basename_colors', 'basename_palette

In [5]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/basename_ref.h5ad"

aggdata = sc.get.aggregate(
    adata,
    by='basename',
    func='sum',
    layer='counts',
)

aggdata.X = aggdata.layers['sum']
del aggdata.layers['sum']
print(f"{aggdata.shape=}")
aggdata.write(outpath)
aggdata.obs.head()


aggdata.shape=(8, 28538)
CPU times: user 5.31 s, sys: 4.93 s, total: 10.2 s
Wall time: 10.3 s


,basename
endothelial_cells,endothelial_cells
fibroblasts,fibroblasts
hematopoietic_progenitors,hematopoietic_progenitors
hsc,hsc
innate_lymphoid_cells,innate_lymphoid_cells


In [6]:
%%time
outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/cell_type_ref.h5ad"

aggdata = sc.get.aggregate(
    adata,
    by='cell_type',
    func='sum',
    layer='counts',
)

aggdata.X = aggdata.layers['sum']
del aggdata.layers['sum']
print(f"{aggdata.shape=}")
aggdata.write(outpath)
aggdata.obs.head()


aggdata.shape=(188, 28538)
CPU times: user 5.93 s, sys: 5.69 s, total: 11.6 s
Wall time: 11.9 s


,cell_type
B-1 B cell,B-1 B cell
B-1a B cell,B-1a B cell
B-1b B cell,B-1b B cell
B-2 B cell,B-2 B cell
CD1c-positive myeloid dendritic cell,CD1c-positive myeloid dendritic cell
